In [21]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import datetime
import time
import os

os.makedirs("data", exist_ok=True)
print("All imports done!")

All imports done!


In [2]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

my_categories = {
    "mobiles":    "https://www.shopsy.in/search?q=smartphones&page={}",
    "laptops":    "https://www.shopsy.in/search?q=laptops&page={}",
    "headphones": "https://www.shopsy.in/search?q=headphones&page={}",
}
pages_to_scrape = 5
wait_time = 3

print("Settings ready!")

Settings ready!


In [3]:
def clean_price(text):
    if text:
        return (
            text.replace("₹", "")
                .replace(",", "")
                .strip()
        )
    return None

def calculate_discount(selling_price, mrp):
    try:
        sp = float(selling_price)
        mp = float(mrp)
        if mp > 0 and mp >= sp:
            return round(((mp - sp) / mp) * 100, 2)
    except:
        pass
    return None

print("Helper functions ready!")

Helper functions ready!


In [4]:
def scrape_shopsy_page(url, category):
    products = []

    try:
        r = requests.get(url, headers=headers, timeout=15)
        r.raise_for_status()
    except Exception as e:
        print(f"Failed: {e}")
        return products

    soup = BeautifulSoup(r.content, "lxml")

   
    cards = soup.find_all("div", class_="r-1ah4tor")
    if not cards:
        cards = soup.find_all("div", class_="r-1hvjb8t")
    print(f"cards found: {len(cards)}", end=" | ")
    for card in cards:
        try:
            
            name_tag = card.find("div", class_="bkNEtl") or \
           card.find("div", class_=lambda c: c and "r-1ss6j8a" in c)
            product_name = name_tag.get_text().strip() if name_tag else None

            price_tag = card.find("div", class_=lambda c: c and "r-cqee49" in c and "r-1vgyyaa" in c)
            selling_price = clean_price(price_tag.get_text() if price_tag else None)

            mrp_tag = card.find("div", class_=lambda c: c and "r-1h7g6bg" in c and "r-1vgyyaa" in c)
            mrp = clean_price(mrp_tag.get_text() if mrp_tag else None)

            discount_tag = card.find("div", class_=lambda c: c and "r-183gjk9" in c and "r-1vgyyaa" in c)
            discount_shown = None
            if discount_tag:
                discount_shown = (
                    discount_tag.get_text()
                    .replace("% off", "")
                    .replace("%", "")
                    .strip()
                )


            reviews_tag = card.find("div", class_=lambda c: c and "r-9iso6" in c)
            reviews = reviews_tag.get_text().strip() if reviews_tag else None

            if not product_name or not selling_price:
                continue

            real_discount = calculate_discount(selling_price, mrp)

            products.append({
                "product_name":        product_name,
                "selling_price_inr":   selling_price,
                "mrp_inr":             mrp,
                "discount_as_listed":  discount_shown,
                "discount_calculated": real_discount,
                "num_reviews":         reviews,
                "category":            category,
                "platform":            "Shopsy",
                "source_url":          url,
                "scraped_at":          datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

        except Exception:
            continue

    return products

print("Scraping function ready!")

Scraping function ready!


In [32]:
all_products = []

print("Starting Shopsy scrape...\n")

for category, url_template in my_categories.items():
    print(f"\nCategory: {category}")

    for page_num in range(1, pages_to_scrape + 1):
        url = url_template.format(page_num)
        print(f"  Page {page_num}/{pages_to_scrape} ... ", end="")

        page_data = scrape_shopsy_page(url, category)
        all_products.extend(page_data)

        print(f"total so far: {len(all_products)}")
        time.sleep(wait_time)

print(f"\nDone! Total collected: {len(all_products)}")

Starting Shopsy scrape...


Category: mobiles
  Page 1/5 ... cards found: 24 | total so far: 24
  Page 2/5 ... cards found: 24 | total so far: 48
  Page 3/5 ... cards found: 24 | total so far: 72
  Page 4/5 ... cards found: 24 | total so far: 96
  Page 5/5 ... cards found: 24 | total so far: 120

Category: laptops
  Page 1/5 ... Failed: ('Connection aborted.', OSError(22, 'Invalid argument'))
total so far: 120
  Page 2/5 ... cards found: 24 | total so far: 144
  Page 3/5 ... cards found: 24 | total so far: 168
  Page 4/5 ... Failed: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
total so far: 168
  Page 5/5 ... cards found: 24 | total so far: 192

Category: headphones
  Page 1/5 ... Failed: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
total so far: 192
  Page 2/5 ... cards found: 40 | total so far: 226
  Page 3/5 ... cards found: 40 | total so far: 266
  Page 4/5 ... Failed: ('Connection aborted.', OSError(22, 'Invalid arg

In [33]:
df_shopsy = pd.DataFrame(all_products)

before = len(df_shopsy)
df_shopsy.drop_duplicates(
    subset=["product_name", "selling_price_inr", "category"],
    inplace=True
)
after = len(df_shopsy)

today = datetime.datetime.now().strftime("%d%b%Y")
filename = f"data/shopsy_raw_{today}.csv"
df_shopsy.to_csv(filename, index=False, encoding="utf-8-sig")

print(f"Rows collected      : {before}")
print(f"After deduplication : {after}")
print(f"Saved to            : {filename}")
df_shopsy.head(10)

Rows collected      : 301
After deduplication : 213
Saved to            : data/shopsy_raw_21Apr2026.csv


,product_name,selling_price_inr,mrp_inr,discount_as_listed,discount_calculated,num_reviews,category,platform,source_url,scraped_at
0,"MOTOROLA g35 5G (Leaf Green, 128 GB)",12489,12499,None,0.08,"(1,34,956)",mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=1,2026-04-21 17:20:35
1,"MOTOROLA g06 power (Pantone tapestry, 64 GB)",9989,9999,None,0.10,"(14,251)",mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=1,2026-04-21 17:20:35
2,"MOTOROLA g35 5G (Midnight Black, 128 GB)",12489,12499,None,0.08,"(1,34,956)",mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=1,2026-04-21 17:20:35
3,"Google Pixel 9A (Obsidian, 256 GB)",39989,49999,20,20.02,"(10,197)",mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=1,2026-04-21 17:20:35
4,"Apple iPhone 15 (Pink, 128 GB)",59890,59900,None,0.02,"(2,74,062)",mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=1,2026-04-21 17:20:35
5,"MOTOROLA g57 power 5G (Pantone Regatta, 128 GB)",15989,17999,11,11.17,"(40,902)",mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=1,2026-04-21 17:20:35
6,"Samsung Galaxy F06 5G (Bahama Blue, 128 GB)",13989,15499,9,9.74,"(24,157)",mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=1,2026-04-21 17:20:35
7,"vivo T5x 5G (Cyber Green, 128 GB)",24989,30999,19,19.39,"(8,294)",mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=1,2026-04-21 17:20:35
8,"vivo T5x 5G (Cyber Green, 128 GB)",22989,28999,20,20.72,"(7,458)",mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=1,2026-04-21 17:20:35
9,"vivo T5x 5G (Star Silver, 128 GB)",22989,28999,20,20.72,"(7,458)",mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=1,2026-04-21 17:20:35


In [34]:
print("=== SHOPSY DATASET SUMMARY ===\n")
print(f"Total rows: {len(df_shopsy)}")
print(f"\nRows per category:")
print(df_shopsy["category"].value_counts())
print(f"\nMissing values:")
print(df_shopsy.isnull().sum())
print(f"\nDiscount range:")
print(df_shopsy["discount_calculated"].describe())

=== SHOPSY DATASET SUMMARY ===

Total rows: 213

Rows per category:
mobiles       102
laptops        57
headphones     54
Name: category, dtype: int64

Missing values:
product_name            0
selling_price_inr       0
mrp_inr                 0
discount_as_listed     37
discount_calculated     0
num_reviews             5
category                0
platform                0
source_url              0
scraped_at              0
dtype: int64

Discount range:
count    213.000000
mean      29.598545
std       30.070107
min        0.010000
25%        5.610000
50%       18.950000
75%       54.200000
max       90.600000
Name: discount_calculated, dtype: float64
